# Get data from ECCC weather stations


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import json
from pandas import json_normalize

import os
import warnings
from urllib.parse import urlencode, urljoin
from urllib.request import urlopen

from cartopy import crs as ccrs, feature as cfeature
from datetime import datetime as dt

## Documentation of the ECCC API used below
- Data are organized into "collections" (https://api.weather.gc.ca/collections) and you can see I use the "climate-daily" collection to get Daily weather, and also the "climate-stations" collection to find Stations based on their STN_ID
- I think you can modify this script to pull from the "climate-hourly" collection and get hourly data (https://api.weather.gc.ca/collections/climate-hourly)

## Collection: climate-stations

In [15]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-stations/items/")
query = {"ENG_PROV_NAME": "BRITISH-COLUMBIA",
         "limit":"10000",
        "f":"csv"}
url = f"{api}?{urlencode(query)}"
print(url)

# Send the request to the server
resp = requests.get(url)
#resp

# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    df = pd.read_csv(req)
df.sort_index().head(100)

https://api.weather.gc.ca/collections/climate-stations/items/?ENG_PROV_NAME=BRITISH-COLUMBIA&limit=10000&f=csv


,x,y,STN_ID,STATION_NAME,PROV_STATE_TERR_CODE,ENG_PROV_NAME,FRE_PROV_NAME,COUNTRY,LATITUDE,LONGITUDE,...,LAST_DATE,HLY_FIRST_DATE,HLY_LAST_DATE,DLY_FIRST_DATE,DLY_LAST_DATE,MLY_FIRST_DATE,MLY_LAST_DATE,HAS_MONTHLY_SUMMARY,HAS_NORMALS_DATA,HAS_HOURLY_DATA
0,-123.700000,48.916667,2,CHEMAINUS BARE POINT,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,485500000,-1234200000,...,1979-12-31 00:00:00,NaN,NaN,1979-11-01 00:00:00,1979-12-31 00:00:00,1979-01-01 00:00:00,1979-12-01 00:00:00,Y,N,N
1,-123.350000,48.883333,3,GALIANO SOUTH 2,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,485300000,-1232100000,...,1984-12-01 00:00:00,NaN,NaN,1979-05-01 00:00:00,1984-01-31 00:00:00,1979-01-01 00:00:00,1984-12-01 00:00:00,Y,N,N
2,-123.316667,48.866667,4,GALIANO SOUTH,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,485200000,-1231900000,...,1978-12-31 00:00:00,NaN,NaN,1977-07-01 00:00:00,1978-12-31 00:00:00,1977-01-01 00:00:00,1978-12-01 00:00:00,Y,N,N
3,-123.416667,48.550000,5,SAANICH OLDFIELD NORTH,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,483300000,-1232500000,...,1967-12-01 00:00:00,NaN,NaN,1965-12-01 00:00:00,1967-03-31 00:00:00,1966-01-01 00:00:00,1967-12-01 00:00:00,Y,N,N
4,-123.266944,48.455000,6,VICTORIA PHYLLIS STREET,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,482718000,-1231601000,...,2001-08-31 00:00:00,NaN,NaN,1973-06-01 00:00:00,2001-08-31 00:00:00,1973-01-01 00:00:00,2001-08-01 00:00:00,Y,N,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-123.626408,48.646947,97,SHAWNIGAN LAKE,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,483849010,-1233735070,...,2023-11-02 00:00:00,NaN,NaN,1911-05-01 00:00:00,2023-11-02 00:00:00,1911-01-01 00:00:00,2007-02-01 00:00:00,Y,Y,N
96,-123.633333,48.650000,98,SHAWNIGAN LAKE NORTH,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,483900000,-1233800000,...,1976-12-01 00:00:00,NaN,NaN,1973-02-01 00:00:00,1976-04-30 00:00:00,1973-01-01 00:00:00,1976-12-01 00:00:00,Y,N,N
97,-123.400000,48.683333,99,SIDNEY DEEP COVE,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,484100000,-1232400000,...,1973-12-01 00:00:00,NaN,NaN,1967-04-01 00:00:00,1973-05-31 00:00:00,1967-01-01 00:00:00,1973-12-01 00:00:00,Y,N,N
98,-123.733333,48.366667,100,SOOKE,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,482200000,-1234400000,...,1992-12-01 00:00:00,NaN,NaN,1970-03-01 00:00:00,1992-06-30 00:00:00,1970-01-01 00:00:00,1992-12-01 00:00:00,Y,N,N


In [16]:
df.query("STATION_NAME.str.contains('VANCOUVER')")

,x,y,STN_ID,STATION_NAME,PROV_STATE_TERR_CODE,ENG_PROV_NAME,FRE_PROV_NAME,COUNTRY,LATITUDE,LONGITUDE,...,LAST_DATE,HLY_FIRST_DATE,HLY_LAST_DATE,DLY_FIRST_DATE,DLY_LAST_DATE,MLY_FIRST_DATE,MLY_LAST_DATE,HAS_MONTHLY_SUMMARY,HAS_NORMALS_DATA,HAS_HOURLY_DATA
670,-123.050000,49.316667,676,N VANCOUVER CLOVERLEY,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491900000,-1230300000,...,1984-12-01 00:00:00,NaN,NaN,1968-07-01 00:00:00,1984-04-30 00:00:00,1968-01-01 00:00:00,1984-12-01 00:00:00,Y,N,N
672,-123.016667,49.316667,678,N VANCOUVER SEYMOUR BLVD,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491900000,-1230100000,...,1982-12-01 00:00:00,NaN,NaN,1968-01-01 00:00:00,1982-03-31 00:00:00,1968-01-01 00:00:00,1982-12-01 00:00:00,Y,N,N
673,-123.116667,49.350000,679,N VANCOUVER CAPILANO HIGHLANDS,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,492100000,-1230700000,...,1991-11-01 00:00:00,NaN,NaN,1989-04-01 00:00:00,1991-09-30 00:00:00,1989-01-01 00:00:00,1991-11-01 00:00:00,Y,N,N
674,-123.107222,49.359722,680,N VANCOUVER CLEVELAND,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,492135000,-1230626000,...,2003-09-30 00:00:00,NaN,NaN,1968-04-01 00:00:00,2003-09-30 00:00:00,1968-01-01 00:00:00,2003-09-01 00:00:00,Y,N,N
682,-123.050000,49.283333,688,VANCOUVER GRANDVIEW N,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491700000,-1230300000,...,1963-07-31 00:00:00,NaN,NaN,1961-05-01 00:00:00,1963-07-31 00:00:00,1961-01-01 00:00:00,1962-12-01 00:00:00,Y,N,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707,-123.187236,49.182500,51357,VANCOUVER SEA ISLAND CCG,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491057000,-1231114050,...,2026-10-18 12:30:11,2013-03-20 06:00:00,2026-10-18 12:30:11,2013-04-25 00:00:00,2026-10-18 00:00:00,NaN,NaN,Y,N,Y
1712,-123.183889,49.194722,51442,VANCOUVER INTL A,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491141000,-1231102000,...,2026-10-18 12:30:11,2013-06-11 06:00:00,2026-10-18 12:30:11,2013-06-13 00:00:00,2026-10-18 00:00:00,NaN,NaN,Y,N,Y
1725,-123.183889,49.194722,52218,VANCOUVER INT'L HWOS,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491141000,-1231102000,...,2014-07-04 00:00:00,NaN,NaN,2014-02-21 00:00:00,2014-07-04 00:00:00,NaN,NaN,NaN,NaN,NaN
1758,-123.111389,49.294444,55438,VANCOUVER HARBOUR,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491740000,-1230641000,...,2026-10-18 12:30:11,2023-04-24 09:00:00,2026-10-18 12:30:11,2023-04-24 00:00:00,2026-10-18 00:00:00,NaN,NaN,Y,N,N


## Get temps for October for Vancouver International Airport

### Station 889 doesn't have mean temp data for recent years so need the other station 51442

In [8]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-daily/items/")
query = {
    "datetime": "2026-10-01/..", # all items from 2026-10-05 onward
    "STN_ID": "51442", # the other Vancouver Intl A
    "sortby": "LOCAL_DATE",
    "f": "csv" # what format do I want the response to be in
}
url = f"{api}?{urlencode(query)}"


# Use geopandas to convert the json output to a GeoDataFrame.
with urlopen(url=str(url)) as req:
    df = pd.read_csv(req)
df.sort_index().head(20)

,x,y,STATION_NAME,CLIMATE_IDENTIFIER,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,...,DIRECTION_MAX_GUST_FLAG,SPEED_MAX_GUST,SPEED_MAX_GUST_FLAG,COOLING_DEGREE_DAYS,COOLING_DEGREE_DAYS_FLAG,HEATING_DEGREE_DAYS,HEATING_DEGREE_DAYS_FLAG,MIN_REL_HUMIDITY,MIN_REL_HUMIDITY_FLAG,STN_ID
0,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.1,2026-10-01 00:00:00,BC,2026,10,1,...,M,NaN,M,0.0,NaN,5.1,NaN,81.0,NaN,51442
1,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.2,2026-10-02 00:00:00,BC,2026,10,2,...,M,NaN,M,0.0,NaN,2.7,NaN,87.0,NaN,51442
2,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.3,2026-10-03 00:00:00,BC,2026,10,3,...,M,NaN,M,0.0,NaN,3.4,NaN,89.0,NaN,51442
3,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.4,2026-10-04 00:00:00,BC,2026,10,4,...,M,NaN,M,0.0,NaN,5.4,NaN,94.0,NaN,51442
4,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.5,2026-10-05 00:00:00,BC,2026,10,5,...,M,NaN,M,0.0,NaN,5.4,NaN,92.0,NaN,51442
5,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.6,2026-10-06 00:00:00,BC,2026,10,6,...,M,NaN,M,0.0,NaN,4.7,NaN,95.0,NaN,51442
6,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.7,2026-10-07 00:00:00,BC,2026,10,7,...,M,NaN,M,0.0,NaN,5.4,NaN,94.0,NaN,51442
7,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.8,2026-10-08 00:00:00,BC,2026,10,8,...,M,NaN,M,0.0,NaN,3.9,NaN,76.0,NaN,51442
8,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.9,2026-10-09 00:00:00,BC,2026,10,9,...,NaN,71.0,NaN,0.0,NaN,5.2,NaN,59.0,NaN,51442
9,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.10,2026-10-10 00:00:00,BC,2026,10,10,...,NaN,37.0,NaN,0.0,NaN,8.6,NaN,73.0,NaN,51442


In [11]:
#df.columns

In [13]:
df_red = df[['x', 'y', 'STATION_NAME', 'ID', 'LOCAL_DATE',
       'PROVINCE_CODE', 'LOCAL_YEAR', 'LOCAL_MONTH', 'LOCAL_DAY',
       'MEAN_TEMPERATURE', 'MIN_TEMPERATURE','MAX_TEMPERATURE',]]
df_red

,x,y,STATION_NAME,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,MEAN_TEMPERATURE,MIN_TEMPERATURE,MAX_TEMPERATURE
0,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.1,2026-10-01 00:00:00,BC,2026,10,1,12.9,10.7,15.1
1,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.2,2026-10-02 00:00:00,BC,2026,10,2,15.3,12.7,17.9
2,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.3,2026-10-03 00:00:00,BC,2026,10,3,14.6,11.9,17.3
3,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.4,2026-10-04 00:00:00,BC,2026,10,4,12.6,8.7,16.4
4,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.5,2026-10-05 00:00:00,BC,2026,10,5,12.6,8.7,16.4
5,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.6,2026-10-06 00:00:00,BC,2026,10,6,13.3,9.3,17.3
6,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.7,2026-10-07 00:00:00,BC,2026,10,7,12.6,9.6,15.6
7,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.8,2026-10-08 00:00:00,BC,2026,10,8,14.1,9.3,18.8
8,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.9,2026-10-09 00:00:00,BC,2026,10,9,12.8,10.4,15.1
9,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.10,2026-10-10 00:00:00,BC,2026,10,10,9.4,4.6,14.2


# So, you can probably modify the above to read from climate-hourly once you've found a station in Hamilton, ON
- note, you can go to the website for the collection (https://api.weather.gc.ca/collections/climate-hourly/items?lang=en) to search for the station too. And, you might be able to download a CSV file from the website, instead of using the API as I did above.